# Section 2 – Explore and clean the data
**STUDENT notebook: complete the TODO cells**

| | |
|---|---|
| Connects to | Your ML background (pandas, EDA, leakage) |
| Estimated time | 50 minutes |
| Produces | `data/pokemon_clean.csv` and `data/feature_info.json` |

**Goal:** Understand the table before modelling: missing values, distributions, class balance, correlations. Then decide which columns are features and which are targets.

### How to work in this notebook
- 🟦 **Worked sample** cells run as they are. Read them, run them, change them.
- 🟨 **Your turn** cells contain `# TODO` and `...`. Replace `...` with your code.
- ✅ **Check** cells print `PASS` / `FAIL`. Do not move on while you see a `FAIL`.
- Stuck? Open the same file in the `solution/` folder, but try first.

## Step 0 – Setup

In [ ]:
import importlib.util
import subprocess
import sys

# 1) Make sure the libraries exist in THIS kernel (installs only what is missing)
REQUIRED = {"requests": "requests", "pandas": "pandas", "numpy": "numpy",
            "scikit-learn": "sklearn", "matplotlib": "matplotlib", "pillow": "PIL",
            "joblib": "joblib", "fastapi": "fastapi", "uvicorn": "uvicorn",
            "httpx": "httpx", "pydantic": "pydantic"}
missing = [pkg for pkg, mod in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Installed:", missing, "-> restart the kernel if an import below fails")
else:
    print("All dependencies are available")

# 2) Corporate networks with HTTPS inspection: use the Windows certificate store if available
try:
    import truststore
    truststore.inject_into_ssl()
    print("truststore active (Windows certificate store)")
except ImportError:
    pass

# 3) Lab folders (this notebook lives in student/ or solution/, so the lab root is one level up)
from pathlib import Path
import json

ROOT = Path.cwd().parent
DATA, MODELS, OUT = ROOT / "data", ROOT / "models", ROOT / "outputs"
for folder in (DATA, MODELS, OUT):
    folder.mkdir(exist_ok=True)


def check(name, ok, detail=""):
    """Self-check helper: prints PASS / FAIL."""
    print(("PASS" if ok else "FAIL"), "-", name, ("-> " + str(detail)) if detail else "")


print("Lab root:", ROOT)

## Why this section matters
A model is only as good as the table it learns from. Before training anything we answer four questions:

1. Is anything **missing** or duplicated?
2. How are the two **targets** distributed?
3. Which columns are **related** to the targets?
4. Which columns are **allowed** as features (no leakage)?

We will predict **two columns**:

| Target | Type | Meaning | Models we will compare |
|---|---|---|---|
| `base_experience` | number (regression) | the "power" printed on the card | Linear Regression vs Random Forest |
| `is_legendary` | yes / no (classification) | decides the rarity of the card | Logistic Regression vs Random Forest |

## Step 1 – Load the table from Section 1
If you did not finish Section 1, copy `solution_artifacts/data/pokemon_raw.csv` into the `data/` folder.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(DATA / "pokemon_raw.csv")
print(df.shape)
df.head()

## Step 2 – Missing values, duplicates, data types
🟦 **Worked sample** on a tiny fantasy-heroes table:

In [ ]:
toy = pd.DataFrame({
    "hero":  ["Ayla", "Borin", "Cira", "Dax", "Eld", "Eld"],
    "level": [12, 30, np.nan, 25, 8, 8],
    "hp":    [80, 150, 95, 120, 60, 60],
    "class": ["mage", "tank", "mage", "tank", "rogue", "rogue"],
})
print("missing per column:\n", toy.isna().sum(), sep="")
print("\nduplicated rows:", toy.duplicated().sum())
print("\ndtypes:\n", toy.dtypes, sep="")
toy.describe()

🟨 **Your turn** on the real table `df`.

In [ ]:
# TODO: number of missing values per column, in a Series called `missing`
# HINT: df.isna().sum()
...  # <- write your code here

# TODO: number of fully duplicated rows in `n_duplicates`
...  # <- write your code here

print(missing[missing > 0])
check("type2 has missing values (single-type characters)", missing["type2"] > 0, int(missing["type2"]))
check("no duplicated rows", n_duplicates == 0, n_duplicates)

**Think:** `type2` is missing for many rows. Is that a problem? (Hint: a character may have only one type.)
`base_experience` has *no* missing values in today's data, but we still guard against it in Step 5, because live data changes.

## Step 3 – The two targets
🟦 **Worked sample:** class proportions in the toy table.

In [ ]:
print(toy["class"].value_counts(normalize=True).round(2))
print("mean hp:", toy["hp"].mean(), " median hp:", toy["hp"].median())

In [ ]:
# TODO: percentage of legendary characters (0-100) in `legend_pct`
# HINT: a boolean column averages to the proportion of True
...  # <- write your code here

# TODO: mean and median of base_experience (ignore the missing values) in `xp_mean`, `xp_median`
...  # <- write your code here

print(f"legendary: {legend_pct:.1f}%   mythical: {df['is_mythical'].mean() * 100:.1f}%")
print(f"base_experience mean={xp_mean:.1f} median={xp_median:.1f}")
check("legendary is a minority class (< 15%)", legend_pct < 15, f"{legend_pct:.1f}%")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
df["base_experience"].plot.hist(bins=30, ax=ax[0], color="#0066cc")
ax[0].set_title("base_experience (regression target)")
df["is_legendary"].value_counts().plot.bar(ax=ax[1], color=["#999999", "#f0b400"])
ax[1].set_title("is_legendary (classification target)")
plt.tight_layout()
plt.show()

**Think:** if a model always answers "not legendary", what accuracy does it get? Keep that number: it is the trap we avoid in Section 4.

## Step 4 – Correlations
🟦 **Worked sample:** correlation matrix of the toy table.

In [ ]:
toy[["level", "hp"]].corr().round(2)

In [ ]:
NUMERIC = ["hp", "attack", "defense", "special_attack", "special_defense", "speed", "height", "weight", "generation",
           "base_experience"]

# TODO: correlation matrix of the NUMERIC columns in `corr`
# HINT: df[NUMERIC].corr()
...  # <- write your code here

# TODO: the 5 columns most correlated (absolute value) with base_experience, in a Series `top5`
# HINT: corr["base_experience"].drop("base_experience").abs().sort_values(ascending=False).head(5)
...  # <- write your code here

print(top5.round(2))
check("top5 has 5 values", len(top5) == 5)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(NUMERIC)), NUMERIC, rotation=90)
ax.set_yticks(range(len(NUMERIC)), NUMERIC)
fig.colorbar(im)
ax.set_title("Correlation matrix")
plt.tight_layout()
plt.show()

## Step 5 – Decisions: clean, choose features, save
Decisions we take (write yours in the notebook if you disagree and justify it):

1. **Drop** rows without `base_experience`: we cannot train a regression without its target (it may drop 0 rows today; the rule protects us when the data changes).
2. **Merge** `is_mythical` into `is_legendary`: for the card both are "special" characters. Result: a 0/1 column.
3. **Features** (the inputs of both models), the same 8 for both so the API needs only one input form:
   `hp, attack, defense, special_attack, special_defense, speed, height, weight`.
4. **Not features:** `id`, `name` (identifiers), `type1/type2` (we keep them for the card colors only),
   `generation`, `base_experience` and `is_legendary` (targets must never be inputs).

> ⚠️ **Leakage** = using information at training time that will not exist at prediction time.
> Our API receives stats only, so the model may only use stats.

In [ ]:
FEATURES = ["hp", "attack", "defense", "special_attack", "special_defense", "speed", "height", "weight"]

# TODO: `clean`: a copy of df without the rows where base_experience is missing
# HINT: df.dropna(subset=["base_experience"]).copy()
...  # <- write your code here

# TODO: make base_experience an int and is_legendary a 0/1 int that is 1 when the character is legendary OR mythical
# HINT: (clean["is_legendary"] | clean["is_mythical"]).astype(int)
...  # <- write your code here

clean["type2"] = clean["type2"].fillna("none")
clean = clean.drop(columns=["is_mythical"]).reset_index(drop=True)

print(clean.shape)
check("no missing values in features/targets", clean[FEATURES + ["base_experience", "is_legendary"]].isna().sum().sum() == 0)
check("is_legendary only has 0 and 1", set(clean["is_legendary"].unique()) == {0, 1})
print("legendary rate after merging:", round(clean["is_legendary"].mean() * 100, 1), "%")

Now we save two files: the clean table and a small **JSON contract** with the feature list and the valid range of every
feature. Section 5 uses those ranges to validate API input.

In [ ]:
clean.to_csv(DATA / "pokemon_clean.csv", index=False)

# TODO: dict `info` with: "features" (list), "targets" (dict with the two target names), "rows" (int),
# HINT: "legendary_rate" (float) and "ranges": {feature: {"min": ..., "max": ...}} computed from `clean`
...  # <- write your code here

(DATA / "feature_info.json").write_text(json.dumps(info, indent=2), encoding="utf-8")
print(json.dumps(info["ranges"], indent=1)[:400])
check("clean csv saved", (DATA / "pokemon_clean.csv").exists())
check("info has all keys", {"features", "targets", "rows", "legendary_rate", "ranges"} <= set(info))

## Questions
1. Why do we keep the same 8 features for both models?
2. Name one column that would be **leakage** if used as a feature for `is_legendary`. (Think about what the API receives.)
3. Which two features are the most correlated with `base_experience`? Does it match your intuition about a game?

### Extra challenge
Add a column `total_stats = hp + attack + ... + speed`. Would it help a **linear** model? Would it help a **random forest**?
(You can test it in Section 3.)

**Next:** Section 3 – train the regression models.